## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

## WEEK 2 BIG CHALLENGE
- Start by coming up with 3 clarifying questions based on the query
- Tune the searches taking into account the clarifications
- Make the manager an agent with agents as tools and handoffs

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
import gradio as gr

In [2]:
load_dotenv(override=True)

True

In [5]:
class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        description="Exactly 3 clarifying questions to disambiguate the research query.",
        min_length=3,
        max_length=3,
    )

INSTRUCTIONS = """You are a helpful research assistant. Analyze the user's research query and produce exactly 3 clarifying questions that would help narrow or focus the research.
Consider scope (e.g. time range, geography), depth (e.g. overview vs technical), and focus (e.g. audience, use case). Be concise and direct."""

clarifier_agent = Agent(
    name="ClarifierAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ClarifyingQuestions,
)

In [6]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send an email with the given subject and HTML body"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("kelvinator4leo@gmail.com")  # Change to your verified sender
    to_email = To("isievwore.kelvin@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print("Email response", response.status_code)
    return "success"


INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
    handoff_description="Format the report as HTML and send it by email",
)


In [7]:

INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [8]:
HOW_MANY_SEARCHES = 5

INSTRUCTIONS = f"""You are a helpful research assistant. Given a query and optional user clarifications, come up with a set of web searches to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
Use any user clarifications (e.g. time frame, region, audience, depth) to narrow or focus the search plan."""


class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")
    
planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [9]:
INSTRUCTIONS = (
    "You are a research assistant. Given a search term, you search the web for that term and "
    "produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 "
    "words. Capture the main points. Write succintly, no need to have complete sentences or good "
    "grammar. This will be consumed by someone synthesizing a report, so its vital you capture the "
    "essence and ignore any fluff. Do not include any additional commentary other than the summary itself."
)

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [10]:
planner_tool = planner_agent.as_tool(
    tool_name="planner",
    tool_description="Plan web searches for a research query. Include the user's query and their clarifications in your input.",
)
search_tool = search_agent.as_tool(
    tool_name="search",
    tool_description="Run one web search for a given search term and reason. Input should be 'Search term: ... Reason: ...'.",
)
writer_tool = writer_agent.as_tool(
    tool_name="writer",
    tool_description="Write a detailed markdown report from the original query and a list of search result summaries.",
)

RESEARCH_MANAGER_INSTRUCTIONS = """You are a Research Manager. Your goal is to produce a research report and hand it off to the Email agent.

Follow these steps:
1. Use the planner tool with the user's query and clarifications to get a search plan.
2. For each search in the plan, use the search tool (pass the search term and reason).
3. Use the writer tool with the original query and all search summaries (combine the search results into one input).
4. Hand off the resulting report (the markdown report) to the Email agent. Do not send the email yourself.

Crucial: You must use the planner, then search for each item in the plan, then the writer, then hand off exactly once to the Email agent."""

research_manager = Agent(
    name="Research Manager",
    instructions=RESEARCH_MANAGER_INSTRUCTIONS,
    tools=[planner_tool, search_tool, writer_tool],
    handoffs=[email_agent],
    model="gpt-4o-mini",
)

In [11]:
async def get_clarifying_questions(query: str) -> str:
    """Run the clarifier agent and return the 3 questions as formatted text."""
    result = await Runner.run(clarifier_agent, query)
    data = result.final_output_as(ClarifyingQuestions)
    return "\n".join(f"{i+1}. {q}" for i, q in enumerate(data.questions))


async def run_research(query: str, clarifications: str):
    """Run the research manager with query and clarifications; yields trace link then report."""
    message = f"Query: {query}\nClarifications: {clarifications}" if clarifications.strip() else f"Query: {query}\nClarifications: (none provided)"
    trace_id = gen_trace_id()
    with trace("Research trace", trace_id=trace_id):
        yield f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}"
        result = await Runner.run(research_manager, message)
        yield result.final_output or "Research complete (report sent by email)."

In [12]:
with gr.Blocks(theme=gr.themes.Default(primary_hue="sky")) as ui:
    gr.Markdown("# Deep Research")
    query_textbox = gr.Textbox(label="What topic would you like to research?")
    get_questions_btn = gr.Button("Get clarifying questions", variant="secondary")
    questions_display = gr.Markdown(label="Clarifying questions (answer these below to tune the search)")
    clarifications_textbox = gr.Textbox(
        label="Your clarifications (optional — paste answers to the questions above to tune the search)",
        placeholder="e.g. Focus on 2025–2026. Audience: technical leads. Region: Africa.",
        lines=3,
    )
    run_button = gr.Button("Run research", variant="primary")
    report = gr.Markdown(label="Report")

    get_questions_btn.click(fn=get_clarifying_questions, inputs=query_textbox, outputs=questions_display)
    run_button.click(fn=run_research, inputs=[query_textbox, clarifications_textbox], outputs=report)
    query_textbox.submit(fn=get_clarifying_questions, inputs=query_textbox, outputs=questions_display)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Email response 202


### As always, take a look at the trace

https://platform.openai.com/traces